In [1]:
import warnings
warnings.filterwarnings('ignore')
import langchain_community
from langchain_community.document_loaders import PyPDFLoader
loder = PyPDFLoader("India_Encyclopedia_100_Pages_Rebuilt.pdf")
pages = loder.load()


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
spliter = RecursiveCharacterTextSplitter(chunk_size = 1800 , chunk_overlap = 250)
texts = spliter.split_documents(pages)
chunks = []
for i in texts:
    chunks.append(i.page_content)
metadata = []
for i in texts:
    metadata.append(i.metadata)



In [ ]:
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
embedding_function = SentenceTransformerEmbeddingFunction()
client = chromadb.PersistentClient(path="./Databse_VectorDb")
collenction = client.get_or_create_collection(name="Collection_DB",embedding_function=embedding_function)

try:
    if collenction.count()==0:
        collenction.add(
            documents=chunks,
            ids = [str(i) for i in range(len(chunks))],
            metadatas=metadata
        )
except Exception as e:
    print(str(e))
collenction.count()


100

In [ ]:
from langchain_groq import ChatGroq

import os 
from dotenv import load_dotenv
load_dotenv()
api = os.getenv("GRQO_QPI_KEY")

grqo_llm_model = ChatGroq(model="openai/gpt-oss-120b",api_key=api)


In [ ]:
from rank_bm25 import BM25Okapi
def token_create(i):
    i = i.lower()
    i = i.split()
    return i
token = [token_create(i) for i in chunks]
token_for_keyword_search = BM25Okapi(token)


In [ ]:
def retrival(query:str):
    prompt = f"""Write the query omly for symentic search : {query}"""
    query_rewrite = grqo_llm_model.invoke(prompt).content
    response = collenction.query(query_texts=[query_rewrite],n_results=5)
    document = response['documents'][0]
    distance = response['distances'][0]

    print(distance)
    thresold = 1.0
    near_chunks = []
    for i , j in zip(distance,document):
        if thresold>i:
           near_chunks.append(j)

    score = token_for_keyword_search.get_scores(token_create(query_rewrite))

    def near_index_find(score,k=10):
        index = list(enumerate(score))
        index_sorted = sorted(index,key=lambda x:x[1],reverse=True)
        return [inx for inx , sc in index_sorted[:k]]
    
    get_index = near_index_find(score,k=10)

    index_to_chunk_convert=[]
    for i in get_index:
        index_to_chunk_convert.append(chunks[i])

    rrf_item ={}

    for rank , doc in enumerate(near_chunks):
        rrf_item[doc] = rrf_item.get(doc,0)+1/(rank+60)
    for rank , doc in enumerate(index_to_chunk_convert):
        rrf_item[doc] = rrf_item.get(doc,0)+1/(rank+60)

    merge = sorted(rrf_item.items(),key=lambda x:x[1],reverse=True)

    top_token = [doc for doc , _ in merge[:5]]
    if not top_token:
       return "NOT RELATED CONTENT"
    return "\n\n".join(top_token)


In [ ]:
qu = "what is the largest city in the india ?"
response = retrival(qu)
response[:-1]


[0.444923996925354, 0.4687211513519287, 0.4771173596382141, 0.4779292345046997, 0.48438531160354614]


"INDIA — 100 PAGE ENCYCLOPEDIA\n40\n38 — State — Rajasthan\nA state profile should be studied through capital, language, geography, economy and culture.\nCapital and language\nCapital: Jaipur. Major language(s): Hindi/Rajasthani languages.\nGeography and identity\nIndia's largest state by area, famous for deserts, forts, palaces and colourful folk traditions.\nImportant places\nJaipur, Jodhpur, Udaipur, Jaisalmer\nEconomy\nTourism, minerals, textiles, handicrafts\nQuick revision: Capital and language • Geography and identity • Important places • Economy\n\nINDIA — 100 PAGE ENCYCLOPEDIA\n45\n43 — State — Uttar Pradesh\nA state profile should be studied through capital, language, geography, economy and culture.\nCapital and language\nCapital: Lucknow. Major language(s): Hindi and Urdu.\nGeography and identity\nIndia's most populous state, with major historical, religious, agricultural and industrial centres.\nImportant places\nAgra, Varanasi, Lucknow, Prayagraj\nEconomy\nAgriculture, man

In [31]:
question = "india's richest state ?"
context = retrival(query=question)

prompt = f"""
you are a reliable ai assistent
so fetch the user's qustions answers based on local document :
content : {context}
question " {question}
"""
response = grqo_llm_model.invoke(prompt)
print(response.content)


[0.34556448459625244, 0.47999244928359985, 0.5010607838630676, 0.5188305974006653, 0.5219874382019043]
Based on the excerpts you provided, the document contains information about India’s overall economy, as well as brief profiles of Uttar Pradesh and Rajasthan (including their capitals, languages, geography, important places, and key economic sectors). However, it does **not** specify which Indian state is the richest or provide any ranking of states by wealth, GDP, or per‑capita income.

If you need the answer to “Which state is the richest in India?” you’ll need to consult a source that includes state‑level economic data—such as the latest Indian government statistics on Gross State Domestic Product (GSDP) or reputable economic reports that rank states by per‑capita income. Let me know if you’d like help finding such a source or if there’s another question I can answer using the provided document!
